# 03a — Isolation Forest (Farm A)

Per-turbine anomaly detection with MLflow. Reads **`wind-farm-a-features`**, excludes label columns from training, writes **`wind-farm-a-if-scores`**.

**Farm A (supervised path in 03b)** — **`USABLE_ANOMALY_EVENT_IDS`** in **`02_feature_engineering` CELL 1** lists **all 12** Farm A anomaly events used for supervised LOOCV when predicting toward **anomaly-linked downtime** and the combined **`hours_to_next_operator_warning`** target. **Isolation Forest** remains **per-turbine** and **unsupervised** on **`if_fit_eligible`**; fleet-level supervised regression is in **`03b_xgboost_regressor`**.

**IF vs labeled failures** — Per-turbine IF scores are **novelty** baselines trained on normal rows only; they are **not** required to rank or separate labeled anomaly events unless you evaluate that explicitly.

**Train vs test (`train_test`)** — The competition column can **overlap in calendar time** across splits; the same fault window may appear under both `train` and `prediction`. **Do not** use `train_test == "train"` alone to choose IF `fit()` rows unless `USE_LEGACY_TRAIN_TEST_FOR_FIT = True` (comparison only).

**Clean fit mask** — **`if_fit_eligible`** and **`in_anomaly_window`** are computed in **`02_feature_engineering`** (CELL 8) and stored in the feature Delta table. This notebook only **reads** them so IF, LSTM, and XGBoost share the same baseline. **`TRAIN_END` in CELL 1 below must match `TRAIN_END` in 02 CELL 1** (use CELL 2a here to profile timestamps, then align both notebooks).

**Status history (02 CELL 5)** — **`IF_EXCLUDE_STATUS_HISTORY`** in **CELL 2** (default **`False`**) deliberately **includes** downtime fractions, trip counts, and **`hours_since_last_downtime`** in IF inputs as operational context. Set to **`True`** for a stricter **sensor-only** novelty baseline (exclude those columns from **`feature_cols`**).

**IF scores** — Raw **`if_anomaly_score`** is the sklearn decision function; **`if_anomaly_score_norm`** ∈ [0, 1] is min–max scaled using **only `if_fit_eligible` rows** as the reference range (for Stage 4 ensemble).

**Pre-anomaly / horizon labels** — For supervised early-warning models, add labels in feature engineering or a dedicated notebook; not required for IF but recommended for Stage 3 supervised paths.

**Output table contract** — `wind-farm-a-if-scores` must include at minimum:
`asset_id`, `time_stamp`, `id`, `train_test`, `status_type_id`,
`if_anomaly_score` (float), `if_anomaly_score_norm` (float 0–1), `if_anomaly_flag` (int 0/1),
`if_fit_eligible` (bool), `in_anomaly_window` (bool).

Optional passthrough from the feature table (if present): `risk_score`, `hours_to_next_anomaly`, `hours_to_anomaly_linked_downtime`, `hours_to_next_operator_warning`, and related risk columns (for analysis only — **not** used as IF inputs).

Cell 7 uses **Spark-only** aggregations for feature separation diagnostics (no full-table `toPandas()`); no v2 retrain.


In [ ]:
%pip install mlflow


In [ ]:
dbutils.library.restartPython()


In [ ]:
# CELL 1 — Imports and experiment setup

import mlflow
import mlflow.sklearn
from sklearn.ensemble import IsolationForest
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

CATALOG = "wind-turbine-silver"
FEATURE_TABLE = "wind-farm-a-features"
OUTPUT_TABLE = "wind-farm-a-if-scores"

mlflow.set_experiment(
    "/Users/"
    + spark.sql("SELECT current_user()").first()[0]
    + "/DSMLC-Final-Comp-2026-isolation-forest"
)
mlflow.sklearn.autolog(disable=True)

CONTAMINATION_VALUES = [0.01, 0.03, 0.05]

# Profiling + MLflow documentation only — mask columns come from 02_feature_engineering.
EVENT_TABLE = "wind-farm-a-event-info"
# Must match 02_feature_engineering CELL 1 (source of truth for if_fit_eligible in feature table).
TRAIN_END = "2023-01-01"
USE_LEGACY_TRAIN_TEST_FOR_FIT = False  # True = fit on train_test == "train" only (comparison)


In [ ]:
# CELL 2 — Load and validate (if_fit_eligible / in_anomaly_window from 02_feature_engineering)

df_spark = spark.table(f"`{CATALOG}`.`{FEATURE_TABLE}`")

required = [
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "if_fit_eligible",
    "in_anomaly_window",
]
missing = [c for c in required if c not in df_spark.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing} — run 02_feature_engineering CELL 8 first"
    )

# Metadata + label-derived + mask columns must NOT feed unsupervised IF
# Status history (02 CELL 5): default = include in IF (operational context). Set True to use sensor-only novelty.
IF_EXCLUDE_STATUS_HISTORY = False
STATUS_HISTORY_COLS = [
    "downtime_frac_24h",
    "downtime_frac_7d",
    "downtime_frac_30d",
    "derated_frac_24h",
    "derated_frac_7d",
    "derated_frac_30d",
    "status_change_count_7d",
    "status_change_count_30d",
    "hours_since_last_downtime",
]

EXCLUDE_COLS = {
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "event_id",
    "farm",
    "event_label",
    "event_description",
    "risk_score",
    "hours_to_next_anomaly",
    "hours_to_anomaly_linked_downtime",
    "risk_score_anomaly_linked_downtime",
    "hours_to_next_operator_warning",
    "risk_score_operator_warning",
    "next_anomaly_event_id",
    "hours_to_failure_onset",
    "risk_score_onset",
    "next_failure_onset_ts",
    "next_onset_event_id",
    "next_onset_event_overlaps",
    "in_overlap_event_period",
    "is_usable_supervised_next",
    "if_fit_eligible",
    "in_anomaly_window",
}
if IF_EXCLUDE_STATUS_HISTORY:
    EXCLUDE_COLS |= set(STATUS_HISTORY_COLS)

print(
    "IF status history features:",
    "EXCLUDED (sensor-only IF)" if IF_EXCLUDE_STATUS_HISTORY else "INCLUDED (default — operational context)",
)
feature_cols = [c for c in df_spark.columns if c not in EXCLUDE_COLS]

if len(feature_cols) == 0:
    raise ValueError(
        "No feature columns found — run 02_feature_engineering first"
    )

print(f"Feature count (IF inputs): {len(feature_cols)}")
print(f"Turbine count: {df_spark.select('asset_id').distinct().count()}")
print("Rows in anomaly windows:", df_spark.filter(F.col("in_anomaly_window")).count())
print("Rows if_fit_eligible:", df_spark.filter(F.col("if_fit_eligible")).count())
df_spark.groupBy("train_test", "status_type_id").count().orderBy(
    "train_test", "status_type_id"
).show()
df_spark.groupBy("train_test", "if_fit_eligible").count().orderBy(
    "train_test", "if_fit_eligible"
).show()


In [ ]:
# CELL 2a — Timestamp profiling (run once; set TRAIN_END in 02 CELL 1 and CELL 1 above to match)

print("Timestamp range by train_test split:")
df_spark.groupBy("train_test").agg(
    F.min("time_stamp").alias("min_ts"),
    F.max("time_stamp").alias("max_ts"),
    F.count("*").alias("row_count"),
).orderBy("train_test").show(truncate=False)

ev = spark.table(f"`{CATALOG}`.`{EVENT_TABLE}`")
if "asset" in ev.columns and "asset_id" not in ev.columns:
    ev = ev.withColumnRenamed("asset", "asset_id")

print("\nAnomaly event start/end (choose TRAIN_END before earliest relevant anomaly, e.g. >=24h prior):")
ev.filter(F.col("event_label") == "anomaly").select(
    "asset_id", "event_start", "event_end", "event_description"
).orderBy("event_start").show(truncate=False)

In [ ]:
# CELL 3 — Training loop (one model per turbine, nested MLflow runs)

turbines = sorted(
    [r.asset_id for r in df_spark.select("asset_id").distinct().collect()]
)
print(f"Training Isolation Forest for {len(turbines)} turbines ...\n")

all_scores = []

with mlflow.start_run(run_name="IF_Farm_A_parent"):
    mlflow.log_param("n_turbines", len(turbines))
    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_param("feature_table", FEATURE_TABLE)
    mlflow.log_param("contamination_values", str(CONTAMINATION_VALUES))
    mlflow.log_param("TRAIN_END", TRAIN_END)
    mlflow.log_param("if_fit_mask_source", "02_feature_engineering")
    mlflow.log_param("USE_LEGACY_TRAIN_TEST_FOR_FIT", USE_LEGACY_TRAIN_TEST_FOR_FIT)

    for idx, asset_id in enumerate(turbines):
        turbine_pd = (
            df_spark.filter(F.col("asset_id") == asset_id)
            .orderBy("time_stamp")
            .toPandas()
        )

        if USE_LEGACY_TRAIN_TEST_FOR_FIT:
            train_mask = turbine_pd["train_test"] == "train"
        else:
            train_mask = turbine_pd["if_fit_eligible"].fillna(False).astype(bool)

        if train_mask.sum() == 0:
            raise ValueError(
                f"Turbine {asset_id}: no rows with if_fit_eligible — "
                f"rebuild `wind-farm-a-features` in 02 (CELL 8) or adjust TRAIN_END there "
                f"(legacy train_test rows: {(turbine_pd['train_test'] == 'train').sum()})"
            )

        X_train = turbine_pd.loc[train_mask, feature_cols].fillna(0).values
        X_all = turbine_pd[feature_cols].fillna(0).values

        best_model = None
        best_cont = CONTAMINATION_VALUES[1]
        best_score = np.inf

        with mlflow.start_run(run_name=f"IF_{asset_id}", nested=True):
            for cont in CONTAMINATION_VALUES:
                model = IsolationForest(
                    contamination=cont, random_state=42, n_jobs=-1
                )
                model.fit(X_train)
                val_scores = -model.decision_function(X_train)
                mean_val = float(val_scores.mean())
                mlflow.log_metric(f"mean_score_cont_{cont}", mean_val)

                if mean_val < best_score:
                    best_score = mean_val
                    best_cont = cont
                    best_model = model

            scores = -best_model.decision_function(X_all)

            mlflow.sklearn.log_model(best_model, "isolation_forest")
            mlflow.log_param("asset_id", str(asset_id))
            mlflow.log_param("contamination", best_cont)
            mlflow.log_param("fit_rows", int(train_mask.sum()))
            mlflow.log_param(
                "legacy_train_test_train_rows",
                int((turbine_pd["train_test"] == "train").sum()),
            )
            mlflow.log_metric("mean_score_all", float(scores.mean()))

        turbine_pd["if_anomaly_score"] = scores
        turbine_pd["if_anomaly_flag"] = (
            scores > np.percentile(scores, 95)
        ).astype(int)

        out_cols = [
            "asset_id",
            "time_stamp",
            "id",
            "train_test",
            "status_type_id",
            "if_fit_eligible",
            "in_anomaly_window",
            "if_anomaly_score",
            "if_anomaly_flag",
        ]
        for c in ("risk_score", "hours_to_next_anomaly"):
            if c in turbine_pd.columns:
                out_cols.append(c)

        all_scores.append(turbine_pd[out_cols])

        print(
            f"  [{idx + 1}/{len(turbines)}] Turbine {asset_id}: "
            f"cont={best_cont}, mean_score={scores.mean():.4f}, "
            f"flagged={int(turbine_pd['if_anomaly_flag'].sum())}"
        )

df_scored = pd.concat(all_scores, ignore_index=True)
print(f"\nDone. Total scored rows: {len(df_scored):,}")

# Min-max normalize IF score to [0, 1] using fit-eligible rows only (Stage 4 ensemble)
eligible_scores = df_scored.loc[
    df_scored["if_fit_eligible"].fillna(False).astype(bool),
    "if_anomaly_score",
]
if len(eligible_scores) == 0:
    raise ValueError("No if_fit_eligible rows — cannot define normalization reference range")
score_min = float(eligible_scores.min())
score_max = float(eligible_scores.max())
score_range = score_max - score_min if score_max != score_min else 1.0
df_scored["if_anomaly_score_norm"] = (
    (df_scored["if_anomaly_score"] - score_min) / score_range
).clip(0, 1)

runs = mlflow.search_runs(
    filter_string="tags.mlflow.runName = 'IF_Farm_A_parent'",
    max_results=1,
)
parent_run_id = runs.iloc[0].run_id
with mlflow.start_run(run_id=parent_run_id):
    mlflow.log_param("if_norm_score_min", score_min)
    mlflow.log_param("if_norm_score_max", score_max)
    mlflow.log_param("if_norm_reference_rows", int(len(eligible_scores)))

print(
    f"if_anomaly_score_norm: min={score_min:.6f}, max={score_max:.6f}, ref_rows={len(eligible_scores):,}"
)


In [ ]:
# CELL 4 — Validation against event labels

print("Scored output: train_test × if_fit_eligible (row counts)")
print(
    df_scored.groupby(["train_test", "if_fit_eligible"], dropna=False)
    .size()
    .unstack(fill_value=0)
)
print()

events = spark.table(f"`{CATALOG}`.`wind-farm-a-event-info`").toPandas()
if "asset" in events.columns and "asset_id" not in events.columns:
    events = events.rename(columns={"asset": "asset_id"})

print(f"Event table: {len(events)} events")
print(f"  Labels: {events['event_label'].value_counts().to_dict()}\n")

df_eval = df_scored.merge(
    events[
        [
            "asset_id",
            "event_id",
            "event_label",
            "event_start_id",
            "event_end_id",
            "event_description",
        ]
    ],
    on="asset_id",
    how="inner",
)
df_eval["in_event"] = (df_eval["id"] >= df_eval["event_start_id"]) & (
    df_eval["id"] <= df_eval["event_end_id"]
)
df_eval = df_eval[df_eval["in_event"]].copy()

print(f"Rows inside event windows: {len(df_eval):,}\n")

print("Mean IF score by event label:")
print(
    df_eval.groupby("event_label")["if_anomaly_score"].agg(
        ["mean", "std", "count"]
    )
)

anomaly_rows = df_eval[df_eval["event_label"] == "anomaly"]
if not anomaly_rows.empty:
    print("\nMean IF score by fault type:")
    print(
        anomaly_rows.groupby("event_description")["if_anomaly_score"]
        .mean()
        .sort_values(ascending=False)
    )


In [ ]:
# CELL 7 — Spark-only feature separation diagnostics (no v2 retrain)

print("=" * 70)
print("CELL 7 — Feature separation (three groups: anomaly vs normal windows + pre-downtime)")
print("=" * 70)

print("--- Split / mask diagnostics (full feature table, Spark) ---")
df_spark.groupBy("train_test", "if_fit_eligible").count().orderBy(
    "train_test", "if_fit_eligible"
).show()
print(
    "Rows with in_anomaly_window:",
    df_spark.filter(F.col("in_anomaly_window")).count(),
)

events_spark = spark.table(f"`{CATALOG}`.`wind-farm-a-event-info`")
if "asset" in events_spark.columns and "asset_id" not in events_spark.columns:
    events_spark = events_spark.withColumnRenamed("asset", "asset_id")

ev = events_spark.select(
    F.col("asset_id").alias("ev_asset_id"),
    "event_id",
    "event_label",
    "event_start_id",
    "event_end_id",
    "event_description",
)

tagged = df_spark.join(
    F.broadcast(ev),
    (df_spark["asset_id"] == F.col("ev_asset_id"))
    & (df_spark["id"] >= F.col("event_start_id"))
    & (df_spark["id"] <= F.col("event_end_id")),
    "inner",
).drop("ev_asset_id")

n_tagged = tagged.count()
print(f"Tagged rows in event windows: {n_tagged:,}")

anomaly_df = tagged.filter(F.col("event_label") == "anomaly")
normal_df = tagged.filter(F.col("event_label") == "normal")


def mean_dict_spark(df, cols, batch_size=80):
    # Chunked avg to avoid huge codegen on wide feature sets.
    merged = {}
    for i in range(0, len(cols), batch_size):
        batch = cols[i : i + batch_size]
        row = df.select(*[F.avg(F.col(c)).alias(c) for c in batch]).first()
        if row is None:
            continue
        merged.update(row.asDict())
    return merged


a_dict = mean_dict_spark(anomaly_df, feature_cols)
n_dict = mean_dict_spark(normal_df, feature_cols)

hcol = "hours_to_anomaly_linked_downtime"
if hcol in df_spark.columns:
    pre_dt = df_spark.filter(
        (F.col("status_type_id") == 0)
        & F.col(hcol).isNotNull()
        & (F.col(hcol) > 0)
        & (F.col(hcol) <= 168)
    )
    pre_n = pre_dt.count()
    print(
        f"Pre-downtime proxy rows (normal status, 0<h<=168h to {hcol}): {pre_n:,}"
    )
    pre_dict = mean_dict_spark(pre_dt, feature_cols)
else:
    pre_dict = {}
    print(f"Column {hcol} missing — pre-downtime group skipped")

sep_records = []
for feat in feature_cols:
    a_mean = a_dict.get(feat)
    n_mean = n_dict.get(feat)
    p_mean = pre_dict.get(feat) if pre_dict else None
    am = float(a_mean) if a_mean is not None else float("nan")
    nm = float(n_mean) if n_mean is not None else float("nan")
    pm = float(p_mean) if p_mean is not None else float("nan")
    sep_anom_norm = (float(a_mean or 0) - float(n_mean or 0))
    sep_pre_norm = (
        (float(p_mean or 0) - float(n_mean or 0)) if pre_dict else float("nan")
    )
    sep_records.append(
        {
            "feature": feat,
            "anomaly_window_mean": am,
            "normal_window_mean": nm,
            "pre_transition_mean": pm,
            "separation_anomaly_minus_normal": sep_anom_norm,
            "separation_pre_minus_normal": sep_pre_norm,
            "separation_score": sep_anom_norm,
        }
    )

feature_sep_df = pd.DataFrame(sep_records)
feature_sep_df["abs_anomaly_minus_normal"] = (
    feature_sep_df["separation_anomaly_minus_normal"].abs()
)
feature_sep_df = (
    feature_sep_df.sort_values("abs_anomaly_minus_normal", ascending=False)
    .drop(columns=["abs_anomaly_minus_normal"])
    .reset_index(drop=True)
)

csv_path = "/tmp/if_feature_separation.csv"
feature_sep_df.to_csv(csv_path, index=False)

runs = mlflow.search_runs(
    filter_string="tags.mlflow.runName = 'IF_Farm_A_parent'",
    max_results=1,
)
parent_run_id = runs.iloc[0].run_id
print(f"Parent MLflow run: {parent_run_id}")

with mlflow.start_run(run_id=parent_run_id):
    mlflow.log_artifact(csv_path)

print("\nTop 20 features by |anomaly_minus_normal| (see also pre_transition_mean):")
print(feature_sep_df.head(20).to_string(index=False))
print("\nLogged CSV artifact (three-group columns) to MLflow.")


In [ ]:
# CELL 5 — Output schema validation

required_out = [
    "asset_id",
    "time_stamp",
    "id",
    "train_test",
    "status_type_id",
    "if_fit_eligible",
    "in_anomaly_window",
    "if_anomaly_score",
    "if_anomaly_score_norm",
    "if_anomaly_flag",
]
for col in required_out:
    if col not in df_scored.columns:
        raise ValueError(f"Output missing required column: {col}")

print("Output schema validated.")
print(df_scored.dtypes)
print(f"\nTotal rows: {len(df_scored):,}")
print(f"\nNull counts:\n{df_scored.isnull().sum()}")


In [ ]:
# CELL 6 — Save to Delta

(
    spark.createDataFrame(df_scored)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{CATALOG}`.`{OUTPUT_TABLE}`")
)

saved = spark.table(f"`{CATALOG}`.`{OUTPUT_TABLE}`")
print(f"Saved rows: {saved.count():,}")
saved.groupBy("if_anomaly_flag").count().show()
print("Columns in saved table:", saved.columns)
